Installation et téléchargement données

In [1]:
!pip install wbgapi
import wbgapi as wb

import s3fs

fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

MY_BUCKET = "eleanorpaineau"
print(fs.ls(MY_BUCKET))

NoCredentialsError: Unable to locate credentials

In [ ]:
countries = ["AUS", "AUT", "BEL", "CAN", "CHL", "COL", "CRI", "CZE", "DNK", "EST", "FIN",
    "FRA", "DEU", "GRC", "HUN", "ISL", "IRL", "ISR", "ITA", "JPN", "KOR",
    "LVA", "LTU", "LUX", "MEX", "NLD", "NZL", "NOR", "POL", "PRT", "SVK",
    "SVN", "ESP", "SWE", "CHE", "TUR", "GBR", "USA"]
taux_chomage = wb.data.DataFrame("SL.UEM.TOTL.ZS", economy=countries , time=range(1991, 2024))
pib_hab_ppa = wb.data.DataFrame("NY.GDP.PCAP.PP.KD", economy=countries, time=range(1991, 2024))
rnb_hab_ppa = wb.data.DataFrame("NY.GNP.PCAP.PP.KD", economy=countries, time=range(1991, 2024))

with fs.open(f"{MY_BUCKET}/diffusion/projet/raw/wb_chomage.csv", "w") as f:
    taux_chomage.to_csv(f)
with fs.open(f"{MY_BUCKET}/diffusion/projet/raw/wb_pib_hab_ppa.csv", "w") as f:
    pib_hab_ppa.to_csv(f)
with fs.open(f"{MY_BUCKET}/diffusion/projet/raw/wb_rnb_hab_ppa.csv", "w") as f:
    rnb_hab_ppa.to_csv(f)


taux_chomage=taux_chomage.reset_index(names="economy")
pib_hab_ppa=pib_hab_ppa.reset_index(names="economy")
rnb_hab_ppa=rnb_hab_ppa.reset_index(names="economy")
print(pib_hab_ppa.tail(10))


Mise en forme - taux de chômage

In [ ]:
import pandas as pd

# On transforme notre tableau afin que les colonnes des années deviennent des valeurs de la colonne "year"
df_taux_chomage=taux_chomage.melt(
    id_vars=["economy"],
    var_name="year",
    value_name="Taux chômage"
)

#Tri de notre tableau par ordre alphabétique des codes des pays 
df_taux_chomage=df_taux_chomage.sort_values(by="economy", ascending = True)

#Création d'un tableau qui associe les codes pays à leurs noms complets en anglais
noms = pd.DataFrame({
    "economy": ["AUS", "AUT", "BEL", "CAN", "CHL", "COL", "CRI", "CZE", "DNK", "EST", "FIN",
    "FRA", "DEU", "GRC", "HUN", "ISL", "IRL", "ISR", "ITA", "JPN", "KOR",
    "LVA", "LTU", "LUX", "MEX", "NLD", "NZL", "NOR", "POL", "PRT", "SVK",
    "SVN", "ESP", "SWE", "CHE", "TUR", "GBR", "USA"],

    "pays": ["Australia", "Austria", "Belgium", "Canada", "Chile", "Colombia", "Costa Rica",
    "Czechia", "Denmark", "Estonia", "Finland", "France", "Germany", "Greece",
    "Hungary", "Iceland", "Ireland", "Israel", "Italy", "Japan", "South Korea",
    "Latvia", "Lithuania", "Luxembourg", "Mexico", "Netherlands", "New Zealand",
    "Norway", "Poland", "Portugal", "Slovakia", "Slovenia", "Spain", "Sweden",
    "Switzerland", "Turkey", "United Kingdom", "United States"]
})
#On joint notre tableau pour ajouter les noms des pays, en regroupant par le code pays
df_taux_chomage=df_taux_chomage.merge(noms, on="economy", how="left")
print(df_taux_chomage.head(10))

Mise en forme - rnb/tête en ppa

In [ ]:
# De la même manière, on transforme notre tableau afin que les années soient une variable de la colonne "year"
df_rnb_hab_ppa= rnb_hab_ppa.melt(
    id_vars=["economy"],
    var_name="year",
    value_name="RNB/tête PPA"
)

#Tri du tableau par ordre alphabétique des codes pays
df_rnb_hab_ppa=df_rnb_hab_ppa.sort_values(by="economy", ascending = True)

#Merge des tableaux avec taux de chômage et pib/hab ppa en faisant la jointure par pays et année
df_eco=df_taux_chomage.merge(df_rnb_hab_ppa, on=["economy", "year"], how="right")

print(df_eco['RNB/tête PPA'].count())

Mise en forme - pib/tête en ppa

In [ ]:
# De la même manière, on transforme notre tableau afin que les années soient une variable de la colonne "year"
df_pib_hab_ppa= pib_hab_ppa.melt(
    id_vars=["economy"],
    var_name="year",
    value_name="PIB/tête PPA"
)

#Tri du tableau par ordre alphabétique des codes pays
df_pib_hab_ppa=df_pib_hab_ppa.sort_values(by="economy", ascending = True)

#Merge des tableaux avec taux de chômage et pib/hab ppa en faisant la jointure par pays et année
df_eco=df_eco.merge(df_pib_hab_ppa, on=["economy", "year"], how="right")

#Pour que le tableau soit plus propre, on réorganise simplement les colonnes
df_eco = df_eco[["economy", "pays", "year", "Taux chômage", "RNB/tête PPA", "PIB/tête PPA"]]

#Transformation de la colonne "year" pour n'avoir que l'année et non "YRAAAA" (ex : "YR2004")
df_eco["year"]=df_eco["year"].str.replace("YR","")
print(df_eco.head(10))


Statistiques descriptives 

In [ ]:
df_eco.describe(include="all")

Grâce au tableau des statistiques descriptives ci-dessus, on confirme que nos 38 pays sont bien représentés, nos 33 années de 1991 à 2023 le sont aussi. Il y a bien 38x33=1254 données pour le taux de chômage et le PIB/tête PPA : nos données sont complètes pour ces variables. Néanmoins nous n'avons que 1117 données pour le RNB/tête en PPA, il nous manque des données

Recherche des données manquantes :

In [ ]:
print(df_eco.loc[df_eco['RNB/tête PPA'].isna(), ['economy', 'year']])

Répartition des données économiques d'intérêt

On représente la distribution des moyennes des variables, calculées par économie.

In [ ]:
df_eco.groupby('economy')['RNB/tête PPA'].mean().hist(bins=20)

In [ ]:
df_eco.groupby('economy')['PIB/tête PPA'].mean().hist(bins=20)

In [ ]:
df_eco.groupby('economy')['Taux chômage'].mean().hist(bins=20)

On représente l'évolution de la moyenne annuelle de chaque variable.

In [ ]:
df_eco.groupby('year')['RNB/tête PPA'].mean().plot()

In [ ]:
df_eco.groupby('year')['PIB/tête PPA'].mean().plot()

In [ ]:
df_eco.groupby('year')['Taux chômage'].mean().plot()

Trajectoires selon les pays

In [ ]:
df_eco.pivot(index='year', columns='economy', values='RNB/tête PPA').plot(legend=False, alpha=0.5)

In [ ]:
df_eco.pivot(index='year', columns='economy', values='PIB/tête PPA').plot(legend=False, alpha=0.5)

In [ ]:
df_eco.pivot(index='year', columns='economy', values='Taux chômage').plot(legend=False, alpha=0.5)

Corrélation entre les variables

In [ ]:
df_eco[["Taux chômage", "PIB/tête PPA", "RNB/tête PPA"]].corr()